In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, struct, to_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

# Initialize Spark with Kafka package
spark = SparkSession.builder \
    .appName("FraudDetection") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3"
    ) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c67c737b-822e-4322-9b2b-41d7c7cec885;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 1037ms :: artifacts dl 17ms
	:: modules in u

In [3]:
users_df = spark.read.csv(
    "data/user_table.csv",
    header=True,
    inferSchema=True
)

In [4]:
tx_schema = StructType([
    StructField("tx_id", IntegerType(), True),
    StructField("userId", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("timestamp", DoubleType(), True)
])

In [5]:
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "fraud-detection") \
    .load()

In [1]:
parsed_stream = kafka_stream.select(
    from_json(
        col("value").cast("string"),
        tx_schema
    ).alias("tx")
).select("tx.*")

fraud_stream = parsed_stream.filter(
    col("amount") > 5000
)

NameError: name 'kafka_stream' is not defined

In [7]:
# Enrich Stream with User Details
enriched_fraud = fraud_stream.join(
    users_df,
    "userId"
)

In [8]:
# Format for output Kafka topic
output_stream = enriched_fraud \
    .withColumn(
        "value",
        to_json(struct("*"))
    ) \
    .select("value")

In [ ]:
# Write Stream to fraud-notification Topic
query = output_stream.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("topic", "fraud-notification") \
    .option(
        "checkpointLocation",
        "/workspace/checkpoints"
    ) \
    .start()

query.awaitTermination()

26/06/19 04:42:18 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 04:42:18 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [ ]:
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers=['kafka:9092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

producer.send(
    "fraud-detection",
    {
        "tx_id": 1000,
        "userId": 105,
        "amount": 14000,
        "timestamp": 123456789
    }
)

producer.flush()

print("Sent")

In [ ]:
import socket

print(socket.gethostbyname("kafka"))

In [ ]:
output_stream.printSchema()

In [1]:
print("Hello")

Hello
